# Data Engineering 003
## Subqueries, CTEs, and Window Functions — Superstore Dataset

**Objective:** Analyze sales data using SQL by applying Subqueries, CTEs, and Window Functions to solve business queries.

**Steps:**
1. Load Superstore dataset into a table (`superstore_raw`)
2. Create normalized tables: `customers`, `orders`, `products`
3. Insert data using `SELECT DISTINCT`
4. Apply **Subqueries** to filter data
5. Use **CTEs** to compute aggregations
6. Apply **Window Functions** (`ROW_NUMBER`, `RANK`) for ranking
7. Combine JOIN + CTE + Window Functions for final results
8. Solve business queries and present insights


---
## Step  Setup: Import Libraries & Load Data

In [1]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Load CSV into pandas
df = pd.read_csv('superstore_data/Sample - Superstore.csv', encoding='latin1')
df.columns = [c.strip().replace(' ', '_').replace('-', '_').lower() for c in df.columns]

print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
df.head(3)

Dataset shape: (9994, 21)
,
,Columns:
,['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


---
## Step 1 — Load Raw Data into `superstore_raw` Table

In [2]:
# Load entire dataframe into SQLite as superstore_raw
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

# Verify
result = pd.read_sql("SELECT COUNT(*) AS total_rows FROM superstore_raw", conn)
print('Rows loaded into superstore_raw:')
result

Rows loaded into superstore_raw:


,total_rows
0,9994


In [3]:
# Preview raw table
pd.read_sql("SELECT * FROM superstore_raw LIMIT 5", conn)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


---
## Step 2 — Create Normalized Tables: `customers`, `orders`, `products`

In [4]:
# ── CREATE TABLES ──────────────────────────────────────────────

cursor.executescript("""
-- Customers table
CREATE TABLE IF NOT EXISTS customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT,
    segment       TEXT,
    country       TEXT,
    city          TEXT,
    state         TEXT,
    region        TEXT
);

-- Products table
CREATE TABLE IF NOT EXISTS products (
    product_id   TEXT PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    sub_category TEXT
);

-- Orders table
CREATE TABLE IF NOT EXISTS orders (
    order_id    TEXT,
    order_date  TEXT,
    ship_date   TEXT,
    ship_mode   TEXT,
    customer_id TEXT,
    product_id  TEXT,
    sales       REAL,
    quantity    INTEGER,
    discount    REAL,
    profit      REAL
);
"""
)

print('Tables created: customers, products, orders')

Tables created: customers, products, orders


---
## Step 3 — Insert Data Using `SELECT DISTINCT`

In [5]:
# ── INSERT into customers using SELECT DISTINCT ─────────────────
cursor.execute("""
INSERT OR IGNORE INTO customers (customer_id, customer_name, segment, country, city, state, region)
SELECT DISTINCT
    customer_id,
    customer_name,
    segment,
    country,
    city,
    state,
    region
FROM superstore_raw;
""")

# ── INSERT into products using SELECT DISTINCT ──────────────────
cursor.execute("""
INSERT OR IGNORE INTO products (product_id, product_name, category, sub_category)
SELECT DISTINCT
    product_id,
    product_name,
    category,
    sub_category
FROM superstore_raw;
""")

# ── INSERT into orders ──────────────────────────────────────────
cursor.execute("""
INSERT INTO orders (order_id, order_date, ship_date, ship_mode,
                    customer_id, product_id, sales, quantity, discount, profit)
SELECT
    order_id, order_date, ship_date, ship_mode,
    customer_id, product_id, sales, quantity, discount, profit
FROM superstore_raw;
""")

conn.commit()

# Verify counts
for tbl in ['customers', 'products', 'orders']:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {tbl}", conn)['cnt'][0]
    print(f'{tbl:12s}: {cnt:,} rows')


customers   : 793 rows
,products    : 1,862 rows
,orders      : 9,994 rows


In [6]:
# Preview customers
print('--- customers ---')
display(pd.read_sql("SELECT * FROM customers LIMIT 5", conn))

# Preview products
print('--- products ---')
display(pd.read_sql("SELECT * FROM products LIMIT 5", conn))

# Preview orders
print('--- orders ---')
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

--- customers ---


,customer_id,customer_name,segment,country,city,state,region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South


--- products ---


,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


--- orders ---


,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


---
## Step 4 — Appply Subqueries

### 4A. Orders with Sales Above Average (Subquery in WHERE)

In [7]:
query_5a = """
-- Subquery: Orders where Sales > Average Sales across all orders
SELECT
    o.order_id,
    c.customer_name,
    p.product_name,
    o.sales,
    ROUND((SELECT AVG(sales) FROM orders), 2) AS avg_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
WHERE o.sales > (SELECT AVG(sales) FROM orders)
ORDER BY o.sales DESC
LIMIT 10;
"""

result_5a = pd.read_sql(query_5a, conn)
print('Orders with Sales ABOVE Average:')
result_5a

Orders with Sales ABOVE Average:


,order_id,customer_name,product_name,sales,avg_sales
0,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.480,229.86
1,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.950,229.86
2,CA-2017-140151,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,13999.960,229.86
3,CA-2017-127180,Tom Ashbrook,Canon imageCLASS 2200 Advanced Copier,11199.968,229.86
4,CA-2017-166709,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,10499.970,229.86
5,CA-2016-117121,Adrian Barton,GBC Ibimaster 500 Manual ProClick Binding System,9892.740,229.86
6,CA-2014-116904,Sanjit Chand,Ibico EPK-21 Electric Binding System,9449.950,229.86
7,US-2016-107440,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,229.86
8,CA-2016-158841,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,8749.950,229.86
9,CA-2016-143714,Christopher Conant,Canon imageCLASS 2200 Advanced Copier,8399.976,229.86


### 4B. Customer with the Highest Single Order (Subquery in FROM)

In [8]:
query_5b = """
-- Subquery in FROM: Highest order amount per customer
SELECT
    c.customer_name,
    c.segment,
    c.region,
    top_orders.order_id,
    top_orders.max_order_sales
FROM (
    SELECT
        customer_id,
        order_id,
        MAX(sales) AS max_order_sales
    FROM orders
    GROUP BY customer_id
) AS top_orders
JOIN customers c ON top_orders.customer_id = c.customer_id
ORDER BY top_orders.max_order_sales DESC
LIMIT 10;
"""

result_5b = pd.read_sql(query_5b, conn)
print('Highest Single-Order Sales per Customer (Top 10):')
result_5b

Highest Single-Order Sales per Customer (Top 10):


,customer_name,segment,region,order_id,max_order_sales
0,Sean Miller,Home Office,South,CA-2014-145317,22638.480
1,Tamara Chand,Corporate,West,CA-2016-118689,17499.950
2,Raymond Buch,Consumer,East,CA-2017-140151,13999.960
3,Tom Ashbrook,Home Office,East,CA-2017-127180,11199.968
4,Hunter Lopez,Consumer,Central,CA-2017-166709,10499.970
5,Adrian Barton,Consumer,West,CA-2016-117121,9892.740
6,Sanjit Chand,Consumer,West,CA-2014-116904,9449.950
7,Bill Shonely,Corporate,East,US-2016-107440,9099.930
8,Sanjit Engle,Consumer,East,CA-2016-158841,8749.950
9,Christopher Conant,Consumer,South,CA-2016-143714,8399.976


---
## Step 5 — CTEs (Common Table Expressions)

### 5A. Total Sales Per Customer Using CTE

In [9]:
query_6a = """
-- CTE: Total sales and profit per customer
WITH customer_sales AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id)     AS total_orders,
        ROUND(SUM(sales), 2)         AS total_sales,
        ROUND(SUM(profit), 2)        AS total_profit
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    c.segment,
    c.region,
    cs.total_orders,
    cs.total_sales,
    cs.total_profit
FROM customer_sales cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales DESC
LIMIT 10;
"""

result_6a = pd.read_sql(query_6a, conn)
print('Total Sales Per Customer (Top 10) — via CTE:')
result_6a

Total Sales Per Customer (Top 10) — via CTE:


,customer_name,segment,region,total_orders,total_sales,total_profit
0,Sean Miller,Home Office,South,5,25043.05,-1980.74
1,Tamara Chand,Corporate,West,5,19052.22,8981.32
2,Raymond Buch,Consumer,East,6,15117.34,6976.10
3,Tom Ashbrook,Home Office,East,4,14595.62,4703.79
4,Adrian Barton,Consumer,West,10,14473.57,5444.81
5,Ken Lonsdale,Consumer,Central,12,14175.23,806.85
6,Sanjit Chand,Consumer,West,9,14142.33,5757.41
7,Hunter Lopez,Consumer,Central,6,12873.30,5622.43
8,Sanjit Engle,Consumer,East,11,12209.44,2650.68
9,Christopher Conant,Consumer,South,5,12129.07,2177.05


---
## Step 6 — Window Functions: `ROW_NUMBER` and `RANK`

### 6A. Rank Customers by Total Sales (RANK)

In [11]:
query_7a = """
-- Window Function: RANK customers by total sales
WITH customer_totals AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2)  AS total_sales,
        ROUND(SUM(profit), 2) AS total_profit
    FROM orders
    GROUP BY customer_id
)
SELECT
    RANK() OVER (ORDER BY ct.total_sales DESC)       AS sales_rank,
    ROW_NUMBER() OVER (ORDER BY ct.total_sales DESC) AS row_num,
    c.customer_name,
    c.segment,
    c.region,
    ct.total_sales,
    ct.total_profit
FROM customer_totals ct
JOIN customers c ON ct.customer_id = c.customer_id
ORDER BY sales_rank
LIMIT 15;
"""

result_7a = pd.read_sql(query_7a, conn)
print('Customer Sales Ranking (RANK + ROW_NUMBER):')
result_7a

Customer Sales Ranking (RANK + ROW_NUMBER):


,sales_rank,row_num,customer_name,segment,region,total_sales,total_profit
0,1,1,Sean Miller,Home Office,South,25043.05,-1980.74
1,2,2,Tamara Chand,Corporate,West,19052.22,8981.32
2,3,3,Raymond Buch,Consumer,East,15117.34,6976.10
3,4,4,Tom Ashbrook,Home Office,East,14595.62,4703.79
4,5,5,Adrian Barton,Consumer,West,14473.57,5444.81
5,6,6,Ken Lonsdale,Consumer,Central,14175.23,806.85
6,7,7,Sanjit Chand,Consumer,West,14142.33,5757.41
7,8,8,Hunter Lopez,Consumer,Central,12873.30,5622.43
8,9,9,Sanjit Engle,Consumer,East,12209.44,2650.68
9,10,10,Christopher Conant,Consumer,South,12129.07,2177.05


### 6B. Rank Products Within Each Category (RANK OVER PARTITION BY)

In [12]:
query_7b = """
-- Window Function: Rank products within their category by sales
WITH product_sales AS (
    SELECT
        p.category,
        p.sub_category,
        p.product_name,
        ROUND(SUM(o.sales), 2)  AS total_sales,
        ROUND(SUM(o.profit), 2) AS total_profit
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_id
)
SELECT
    category,
    sub_category,
    product_name,
    total_sales,
    total_profit,
    RANK()       OVER (PARTITION BY category ORDER BY total_sales DESC) AS rank_in_category,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) AS row_in_category
FROM product_sales
ORDER BY category, rank_in_category
LIMIT 18;
"""

result_7b = pd.read_sql(query_7b, conn)
print('Top Products Ranked Within Each Category:')
result_7b

Top Products Ranked Within Each Category:


,category,sub_category,product_name,total_sales,total_profit,rank_in_category,row_in_category
0,Furniture,Chairs,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00,1,1
1,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.97,-669.54,2,2
2,Furniture,Tables,Bretford Rectangular Conference Table Tops,12995.29,-327.23,3,3
3,Furniture,Chairs,Global Troy Executive Leather Low-Back Tilter,12975.38,951.86,4,4
4,Furniture,Bookcases,DMI Eclipse Executive Suite Bookcases,12921.64,19.69,5,5
5,Furniture,Chairs,SAFCO Arco Folding Chair,11572.78,1179.37,6,6
6,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",10637.53,1927.44,7,7
7,Furniture,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,8,8
8,Furniture,Tables,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40,9,9
9,Furniture,Chairs,GuestStacker Chair with Chrome Finish Legs,9070.94,773.26,10,10


---
## Step 7 — Combined: JOIN + CTE + Window Functions

### Final Result: Customer,Total Sales , Rank

In [13]:
query_8 = """
-- Combined: CTE + JOIN + Window Function
-- Final view: customer name, total sales, total profit, rank
WITH customer_summary AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT o.order_id)    AS total_orders,
        ROUND(SUM(o.sales),  2)       AS total_sales,
        ROUND(SUM(o.profit), 2)       AS total_profit,
        ROUND(AVG(o.sales),  2)       AS avg_order_sales
    FROM orders o
    GROUP BY o.customer_id
)
SELECT
    RANK()       OVER (ORDER BY cs.total_sales  DESC) AS overall_rank,
    ROW_NUMBER() OVER (ORDER BY cs.total_profit DESC) AS profit_row,
    c.customer_name,
    c.segment,
    c.region,
    cs.total_orders,
    cs.total_sales,
    cs.total_profit,
    cs.avg_order_sales
FROM customer_summary cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY overall_rank
LIMIT 20;
"""

result_8 = pd.read_sql(query_8, conn)
print('Final Combined Result — Customer: Total Sales, Total Profit, Rank:')
result_8

Final Combined Result — Customer: Total Sales, Total Profit, Rank:


,overall_rank,profit_row,customer_name,segment,region,total_orders,total_sales,total_profit,avg_order_sales
0,1,786,Sean Miller,Home Office,South,5,25043.05,-1980.74,1669.54
1,2,1,Tamara Chand,Corporate,West,5,19052.22,8981.32,1587.68
2,3,2,Raymond Buch,Consumer,East,6,15117.34,6976.10,839.85
3,4,6,Tom Ashbrook,Home Office,East,4,14595.62,4703.79,1459.56
4,5,5,Adrian Barton,Consumer,West,10,14473.57,5444.81,723.68
5,6,123,Ken Lonsdale,Consumer,Central,12,14175.23,806.85,488.80
6,7,3,Sanjit Chand,Consumer,West,9,14142.33,5757.41,642.83
7,8,4,Hunter Lopez,Consumer,Central,6,12873.30,5622.43,1170.30
8,9,13,Sanjit Engle,Consumer,East,11,12209.44,2650.68,642.60
9,10,18,Christopher Conant,Consumer,South,5,12129.07,2177.05,1102.64


---
## Step 8 — Business Queries

### BQ1. Top 10 Customers by Total Sales

In [14]:
bq1 = """
-- Business Query 1: Top 10 customers by total sales
WITH ranked_customers AS (
    SELECT
        o.customer_id,
        ROUND(SUM(o.sales), 2)  AS total_sales,
        ROUND(SUM(o.profit), 2) AS total_profit,
        COUNT(DISTINCT o.order_id) AS order_count,
        RANK() OVER (ORDER BY SUM(o.sales) DESC) AS rnk
    FROM orders o
    GROUP BY o.customer_id
)
SELECT
    rc.rnk           AS rank,
    c.customer_name,
    c.segment,
    c.region,
    rc.order_count,
    rc.total_sales,
    rc.total_profit
FROM ranked_customers rc
JOIN customers c ON rc.customer_id = c.customer_id
WHERE rc.rnk <= 10
ORDER BY rc.rnk;
"""

r_bq1 = pd.read_sql(bq1, conn)
print('BQ1 — Top 10 Customers by Total Sales:')
r_bq1

BQ1 — Top 10 Customers by Total Sales:

,rank,customer_name,segment,region,order_count,total_sales,total_profit
0,1,Sean Miller,Home Office,South,5,25043.05,-1980.74
1,2,Tamara Chand,Corporate,West,5,19052.22,8981.32
2,3,Raymond Buch,Consumer,East,6,15117.34,6976.10
3,4,Tom Ashbrook,Home Office,East,4,14595.62,4703.79
4,5,Adrian Barton,Consumer,West,10,14473.57,5444.81
5,6,Ken Lonsdale,Consumer,Central,12,14175.23,806.85
6,7,Sanjit Chand,Consumer,West,9,14142.33,5757.41
7,8,Hunter Lopez,Consumer,Central,6,12873.30,5622.43
8,9,Sanjit Engle,Consumer,East,11,12209.44,2650.68
9,10,Christopher Conant,Consumer,South,5,12129.07,2177.05


### BQ2. Bottom 10 Customers (Lowest Total Sales)

In [15]:
bq2 = """
-- Business Query 2: Bottom 10 customers — lowest total sales
WITH customer_totals AS (
    SELECT
        o.customer_id,
        ROUND(SUM(o.sales), 2)     AS total_sales,
        ROUND(SUM(o.profit), 2)    AS total_profit,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    GROUP BY o.customer_id
)
SELECT
    ROW_NUMBER() OVER (ORDER BY ct.total_sales ASC) AS bottom_rank,
    c.customer_name,
    c.segment,
    c.region,
    ct.order_count,
    ct.total_sales,
    ct.total_profit
FROM customer_totals ct
JOIN customers c ON ct.customer_id = c.customer_id
ORDER BY ct.total_sales ASC
LIMIT 10;
"""

r_bq2 = pd.read_sql(bq2, conn)
print('BQ2 — Bottom 10 Customers (Lowest Sales):')
r_bq2

BQ2 — Bottom 10 Customers (Lowest Sales):


,bottom_rank,customer_name,segment,region,order_count,total_sales,total_profit
0,1,Thais Sissman,Consumer,West,2,4.83,-3.32
1,2,Lela Donovan,Corporate,Central,1,5.30,0.46
2,3,Carl Jackson,Corporate,East,1,16.52,1.65
3,4,Mitch Gastineau,Corporate,South,1,16.74,-1.25
4,5,Roy Skaria,Home Office,Central,2,22.33,9.58
5,6,Susan Gilcrest,Corporate,Central,3,47.95,-3.71
6,7,Ricardo Emerson,Consumer,East,1,48.36,6.04
7,8,Larry Blacks,Consumer,Central,3,50.19,18.65
8,9,Adrian Shami,Home Office,East,2,58.82,21.85
9,10,Jasper Cacioppo,Consumer,West,4,71.26,-0.36


### BQ3. Single-Order Customers

In [16]:
bq3 = """
-- Business Query 3: Customers who placed only ONE order
WITH order_counts AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(sales), 2)     AS total_sales
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(DISTINCT order_id) = 1
)
SELECT
    ROW_NUMBER() OVER (ORDER BY oc.total_sales DESC) AS row_num,
    c.customer_name,
    c.segment,
    c.region,
    oc.total_orders,
    oc.total_sales
FROM order_counts oc
JOIN customers c ON oc.customer_id = c.customer_id
ORDER BY oc.total_sales DESC
LIMIT 15;
"""

r_bq3 = pd.read_sql(bq3, conn)
print('BQ3 — Single-Order Customers:')
r_bq3

BQ3 — Single-Order Customers:


,row_num,customer_name,segment,region,total_orders,total_sales
0,1,Jenna Caffey,Consumer,West,1,1058.11
1,2,Susan MacKendrick,Consumer,East,1,1043.04
2,3,Theresa Coyne,Corporate,East,1,1038.26
3,4,Jocasta Rupert,Consumer,South,1,863.88
4,5,Patricia Hirasaki,Home Office,South,1,729.65
5,6,Anthony O'Donnell,Corporate,West,1,161.28
6,7,Roland Murray,Consumer,East,1,98.35
7,8,Anemone Ratner,Consumer,South,1,88.15
8,9,Ricardo Emerson,Consumer,East,1,48.36
9,10,Mitch Gastineau,Corporate,South,1,16.74


### BQ4. Orders with Above-Average Sales (Subquery)

In [17]:
bq4 = """
-- Business Query 4: All orders with above-average sales using subquery
SELECT
    o.order_id,
    c.customer_name,
    c.region,
    p.category,
    p.product_name,
    ROUND(o.sales, 2)                                    AS order_sales,
    ROUND((SELECT AVG(sales) FROM orders), 2)            AS avg_sales,
    ROUND(o.sales - (SELECT AVG(sales) FROM orders), 2)  AS above_avg_by
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
WHERE o.sales > (SELECT AVG(sales) FROM orders)
ORDER BY o.sales DESC
LIMIT 15;
"""

r_bq4 = pd.read_sql(bq4, conn)
print('BQ4 — Orders with Above-Average Sales:')
r_bq4

BQ4 — Orders with Above-Average Sales:


,order_id,customer_name,region,category,product_name,order_sales,avg_sales,above_avg_by
0,CA-2014-145317,Sean Miller,South,Technology,Cisco TelePresence System EX90 Videoconferenci...,22638.48,229.86,22408.62
1,CA-2016-118689,Tamara Chand,West,Technology,Canon imageCLASS 2200 Advanced Copier,17499.95,229.86,17270.09
2,CA-2017-140151,Raymond Buch,East,Technology,Canon imageCLASS 2200 Advanced Copier,13999.96,229.86,13770.10
3,CA-2017-127180,Tom Ashbrook,East,Technology,Canon imageCLASS 2200 Advanced Copier,11199.97,229.86,10970.11
4,CA-2017-166709,Hunter Lopez,Central,Technology,Canon imageCLASS 2200 Advanced Copier,10499.97,229.86,10270.11
5,CA-2016-117121,Adrian Barton,West,Office Supplies,GBC Ibimaster 500 Manual ProClick Binding System,9892.74,229.86,9662.88
6,CA-2014-116904,Sanjit Chand,West,Office Supplies,Ibico EPK-21 Electric Binding System,9449.95,229.86,9220.09
7,US-2016-107440,Bill Shonely,East,Technology,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.93,229.86,8870.07
8,CA-2016-158841,Sanjit Engle,East,Technology,HP Designjet T520 Inkjet Large Format Printer ...,8749.95,229.86,8520.09
9,CA-2016-143714,Christopher Conant,South,Technology,Canon imageCLASS 2200 Advanced Copier,8399.98,229.86,8170.12


---
## Step 10 — Brief Insights & Summary

In [18]:
# ── Overall dataset summary ─────────────────────────────────────
summary_q = """
SELECT
    COUNT(DISTINCT customer_id)       AS unique_customers,
    COUNT(DISTINCT order_id)          AS unique_orders,
    COUNT(DISTINCT product_id)        AS unique_products,
    ROUND(SUM(sales), 2)              AS total_revenue,
    ROUND(AVG(sales), 2)              AS avg_order_sales,
    ROUND(SUM(profit), 2)             AS total_profit,
    ROUND(SUM(profit) * 100.0
          / SUM(sales), 2)            AS profit_margin_pct
FROM orders;
"""
summary = pd.read_sql(summary_q, conn)
print('=== DATASET SUMMARY ===')
summary

=== DATASET SUMMARY ===


,unique_customers,unique_orders,unique_products,total_revenue,avg_order_sales,total_profit,profit_margin_pct
0,793,5009,1862,2297200.86,229.86,286397.02,12.47


In [19]:
# ── Single-order customer count ─────────────────────────────────
single = pd.read_sql("""
    SELECT COUNT(*) AS single_order_customers
    FROM (
        SELECT customer_id
        FROM orders
        GROUP BY customer_id
        HAVING COUNT(DISTINCT order_id) = 1
    )
""", conn)['single_order_customers'][0]

top_customer = pd.read_sql("""
    SELECT c.customer_name, ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id ORDER BY total_sales DESC LIMIT 1
""", conn)

avg_sales = pd.read_sql("SELECT ROUND(AVG(sales),2) AS avg FROM orders", conn)['avg'][0]

above_avg_count = pd.read_sql(f"""
    SELECT COUNT(*) AS cnt FROM orders WHERE sales > {avg_sales}
""", conn)['cnt'][0]

print('=== KEY BUSINESS INSIGHTS ===')
print(f"""
1. TOP CUSTOMER   : {top_customer['customer_name'].values[0]} 
                    with ${top_customer['total_sales'].values[0]:,.2f} total sales.

2. ABOVE-AVG ORDERS: {above_avg_count:,} orders have sales above the average 
                     of ${avg_sales:,.2f} per order.

3. SINGLE-ORDER CUSTOMERS: {single} customers placed only one order — 
                           a key target for retention campaigns.

4. WINDOW FUNCTIONS helped rank customers globally (RANK) and within 
   segments (PARTITION BY), revealing concentration of top revenue 
   among a small group.

5. CTEs simplified multi-step aggregations (totals → ranking) 
   making queries readable and reusable.
""")

=== KEY BUSINESS INSIGHTS ===
,
,1. TOP CUSTOMER   : Sean Miller 
,                    with $25,043.05 total sales.
,
,2. ABOVE-AVG ORDERS: 2,360 orders have sales above the average 
,                     of $229.86 per order.
,
,3. SINGLE-ORDER CUSTOMERS: 12 customers placed only one order — 
,                           a key target for retention campaigns.
,
,4. WINDOW FUNCTIONS helped rank customers globally (RANK) and within 
,   segments (PARTITION BY), revealing concentration of top revenue 
,   among a small group.
,
,5. CTEs simplified multi-step aggregations (totals → ranking) 
,   making queries readable and reusable.
,


---
## Conclusion

| Technique | Used For |
|-----------|----------|
| **Subquery (WHERE)** | Filter orders above average sales |
| **Subquery (FROM)** | Derive highest order per customer |
| **CTE** | Aggregate total sales/profit per customer & category |
| **ROW_NUMBER()** | Assign unique sequential row IDs |
| **RANK()** | Rank customers allowing ties |
| **RANK() OVER PARTITION BY** | Rank products within each category |
| **JOIN + CTE + Window** | Final combined customer leaderboard |

All queries were run on a normalized schema (`customers`, `products`, `orders`) built from the raw Superstore CSV using `SELECT DISTINCT` inserts.
